<a href="https://colab.research.google.com/github/eddybanshuy/practica1_sistema_recomendacion/blob/main/KNN_P1_SR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="width: 100%; clear: both;">
<div style="float: left; width: 30%;">
<img src="https://utpl.edu.ec/recursos/img/utpl2.png", align="left" width="280" height="120">
</div>
</div>
<div style="float: right; width: 70%;">
<p style="margin: 0; padding-top: 32px; text-align:right; color:#003366; font-size:16px"><u>Sistema de Recomendacion</u></p>
<p style="margin: 0; text-align:right; color:#999999; font-size:17px">Maestría en Inteligencia Artificial Aplicada</p>
<p style="margin: 0; text-align:right; color:#999999; font-size:17px">Maestrante Edison Banshuy</p>
</div>
</div>
<div style="width: 100%; clear: both;">
<div style="width:100%;">&nbsp;</div>

Implementar el algoritmo de recomendación de filtrado colaborativo KNN usando Python y el conjunto de datos de calificaciones de películas (MovieLens Latest).
Link: https://grouplens.org/datasets/movielens/latest/

El sistema deberá ser capaz de generar recomendaciones en función de sus preferencias y las de ítems similares.

Para proceder a desarrollar la práctica, es necesario que realice lo siguiente:

Experimentación con el algoritmo KNN basado en ítems.

La implementación del sistema de recomendación incluye:



*   Cargar el conjunto de datos de calificaciones procesado en Python.
*   Configurar el algoritmo de filtrado colaborativo basado en ítems.
*   Calcular las predicciones utilizando el conjunto de datos de calificaciones.
*   Generar las recomendaciones de películas para un usuario específico.

Prepare un informe que describa: un análisis del conjunto de datos (características), la medida de similaridad utilizada, resultados de experimentos con el algoritmo de FC: predicción y recomendación.

# Práctica 1: Experimentación con un algoritmo de recomendación basado en memoria

In [30]:
#Librerias a utilizar
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

Cargar DATASET

In [31]:
#Cargamos los dataset antes descargados
#En este caso el dataset utilizado es "ml-latest-small.zip"

ratings = pd.read_csv('ratings.csv')
movies = pd.read_csv('movies.csv')


In [32]:
#Verifico el dataset
print(ratings.head())
print(movies.head())

   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  


Como podemos ver tenemos todos los datos correctamente cargados

In [33]:
#A continuacion crep una matriz ya que KNN necesita una matriz donde
#filas son las peliculas, las columnas son los usuarios y los valores son las calificaciones


movie_features = ratings.pivot_table(
    index='movieId',
    columns='userId',
    values='rating'
).fillna(0)

In [34]:
#imprimo para verificar la matriz
print(movie_features.head())

userId   1    2    3    4    5    6    7    8    9    10   ...  601  602  603  \
movieId                                                    ...                  
1        4.0  0.0  0.0  0.0  4.0  0.0  4.5  0.0  0.0  0.0  ...  4.0  0.0  4.0   
2        0.0  0.0  0.0  0.0  0.0  4.0  0.0  4.0  0.0  0.0  ...  0.0  4.0  0.0   
3        4.0  0.0  0.0  0.0  0.0  5.0  0.0  0.0  0.0  0.0  ...  0.0  0.0  0.0   
4        0.0  0.0  0.0  0.0  0.0  3.0  0.0  0.0  0.0  0.0  ...  0.0  0.0  0.0   
5        0.0  0.0  0.0  0.0  0.0  5.0  0.0  0.0  0.0  0.0  ...  0.0  0.0  0.0   

userId   604  605  606  607  608  609  610  
movieId                                     
1        3.0  4.0  2.5  4.0  2.5  3.0  5.0  
2        5.0  3.5  0.0  0.0  2.0  0.0  0.0  
3        0.0  0.0  0.0  0.0  2.0  0.0  0.0  
4        0.0  0.0  0.0  0.0  0.0  0.0  0.0  
5        3.0  0.0  0.0  0.0  0.0  0.0  0.0  

[5 rows x 610 columns]


In [35]:
#Como se pudo ver antes tiene muchos valores vacios
#El Siguiente codigo me permite solo almacenar valores existentes

movie_features_matrix = csr_matrix(movie_features.values)

Modelo KNN

In [36]:
#Realizo la creacion del modelo KNN

model_knn = NearestNeighbors(
    metric='cosine',
    algorithm='brute',
    n_neighbors=6
)

model_knn.fit(movie_features_matrix)

NearestNeighbors(algorithm='brute', metric='cosine', n_neighbors=6)

Que se desea obtener

*   **metric='cosine'**, calcula que tan parecidas son dos peliculas
*   **algorithm='brute'**, compara todas las peliculas entre si
*   **n_neighbors=6**, busca entre la pelicula consultada y las 5 similares a ella






# Funcion para buscar las peliculas similares

In [37]:
#Creo la funcion para recomendar peliculas basado en items

#nombre de funcion a despues a llamar, con el nombre de la pelicula
def recomendar_peliculas(nombre_pelicula):

    movie_index = movies[
        movies['title'].str.contains(nombre_pelicula,
        case=False,
        na=False)
    ].index[0]

#Busco el ID de la pelicula
    movie_id = movies.iloc[movie_index]['movieId']

    query_index = movie_features.index.get_loc(movie_id)

#Consulto a KNN y obtengo las distancias y los indeses de las mismas
    distances, indices = model_knn.kneighbors(
        movie_features.iloc[query_index,:].values.reshape(1,-1),
        n_neighbors=6
    )

    print("Peliculas recomendadas:")

#Se realiza la transformacion de la distancia en similitud
    for i in range(1, len(distances.flatten())):

        movie_idx = movie_features.index[
            indices.flatten()[i]
        ]

        title = movies[
            movies['movieId'] == movie_idx
        ]['title'].values[0]

        similarity = 1 - distances.flatten()[i]

        print(f"{title} (similitud={similarity:.2f})")

Probamos la funcion enviadole el nombre la pelicula

In [38]:
#En este caso la pelicula a buscar la similutud es "Toy Story"

recomendar_peliculas("Toy Story")

Peliculas recomendadas:
Toy Story 2 (1999) (similitud=0.57)
Jurassic Park (1993) (similitud=0.57)
Independence Day (a.k.a. ID4) (1996) (similitud=0.56)
Star Wars: Episode IV - A New Hope (1977) (similitud=0.56)
Forrest Gump (1994) (similitud=0.55)


Como resultado tenemos que Toy Story 2 tiene una similitud mas acertada con un 0,57%, asi como la pelicula de Jurassic Park

# Usuario especifico

In [39]:
#En este caso obtamos por el usuario 10
user_id = 10

user_ratings = ratings[
    ratings['userId'] == user_id
].sort_values(
    by='rating',
    ascending=False
)

print(user_ratings.head())

      userId  movieId  rating   timestamp
1256      10   136020     5.0  1455302192
1236      10    96079     5.0  1455302172
1229      10    91529     5.0  1455302120
1230      10    92259     5.0  1455301553
1205      10    71579     5.0  1455301869


Como podemos ver este usuario a calificado las peliculas

In [40]:
#Realizamos la funcion para obtener peliculas similares usando KNN

def obtener_similares(movie_id, n_recommendations=5):

    if movie_id not in movie_features.index:
        return []

    query_index = movie_features.index.get_loc(movie_id)
#Obtengo la distancion y los indices de las mismas
    distances, indices = model_knn.kneighbors(
        movie_features.iloc[query_index, :].values.reshape(1, -1),
        n_neighbors=n_recommendations + 1
    )

    similares = []

    for i in range(1, len(indices.flatten())):

        similar_movie_id = movie_features.index[
            indices.flatten()[i]
        ]
#Obtengo la similitud
        similarity = 1 - distances.flatten()[i]

        similares.append(
            (similar_movie_id, similarity)
        )

    return similares

In [41]:
#Gerar recomendaciones para el usuario 10
# Peliculas que ya vio el usuario

seen_movies = set(
    user_ratings['movieId']
)

recommendations = {}

# Tomo las 3 peliculas favoritas del usuario 10

top_movies = user_ratings.head(3)

for movie_id in top_movies['movieId']:

    similares = obtener_similares(movie_id)

    for sim_movie_id, score in similares:

        if sim_movie_id not in seen_movies:

            if sim_movie_id in recommendations:
                recommendations[sim_movie_id] += score
            else:
                recommendations[sim_movie_id] = score

In [42]:
#Ordeno las recomendacion
recommended_movies = sorted(
    recommendations.items(),
    key=lambda x: x[1],
    reverse=True
)

Imprmimos los resultados

In [43]:
print("Recomendaciones para el usuario 10")
print("\n")

for movie_id, score in recommended_movies[:10]:

    title = movies[
        movies['movieId'] == movie_id
    ]['title'].values[0]

    print(f"{title} -> similitud acumulada: {score:.2f}")

Recomendaciones para el usuario 10


Inception (2010) -> similitud acumulada: 0.62
Avengers, The (2012) -> similitud acumulada: 0.61
Star Trek Into Darkness (2013) -> similitud acumulada: 0.60
X-Men: First Class (2011) -> similitud acumulada: 0.59
Django Unchained (2012) -> similitud acumulada: 0.58
Prometheus (2012) -> similitud acumulada: 0.58
Man of Steel (2013) -> similitud acumulada: 0.50
Fury (2014) -> similitud acumulada: 0.50
John Wick (2014) -> similitud acumulada: 0.49
John Wick: Chapter Two (2017) -> similitud acumulada: 0.47


Para generar recomendaciones al usuario 10, se identificaron las peliculas con mayor calificacion otorgada por el usuario.

Despues, mediante el algoritmo KNN basado en items y utilizando la similitud coseno, se localizaron peliculas con patrones de evaluacion similares.

Finalmente, se combinaron las recomendaciones obtenidas de las peliculas favoritas del usuario y se excluyeron aquellas que ya haya visto, obteniendo una lista personalizada de peliculas.

# GitHub
